In [5]:
import nbformat as nbf
from bs4 import BeautifulSoup
import os

def converter_html_para_ipynb_v2(arquivo_entrada, arquivo_saida):
    if not os.path.exists(arquivo_entrada):
        print(f"Erro: O arquivo '{arquivo_entrada}' não foi encontrado.")
        return

    with open(arquivo_entrada, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    nb = nbf.v4.new_notebook()
    cells = []

    # Localiza todas as divs que representam células no seu HTML
    # No seu arquivo, elas seguem a classe 'jp-Notebook-cell'
    containers = soup.find_all('div', class_='jp-Notebook-cell')

    if not containers:
        print("Aviso: Nenhuma célula encontrada com o seletor padrão. Tentando busca genérica...")
        containers = soup.find_all('div', class_=lambda x: x and 'jp-Cell' in x)

    for container in containers:
        # 1. Identifica células de Markdown
        if 'jp-MarkdownCell' in container.get('class', []):
            content = container.find('div', class_='jp-MarkdownOutput')
            if content:
                cells.append(nbf.v4.new_markdown_cell(content.get_text().strip()))

        # 2. Identifica células de Código
        elif 'jp-CodeCell' in container.get('class', []):
            # No seu arquivo, o código está dentro de divs com classe 'highlight'
            code_div = container.find('div', class_='highlight')
            if code_div:
                code_text = code_div.get_text().strip()
                cells.append(nbf.v4.new_code_cell(code_text))

    if cells:
        nb['cells'] = cells
        with open(arquivo_saida, 'w', encoding='utf-8') as f:
            nbf.write(nb, f)
        print(f"Sucesso! O arquivo '{arquivo_saida}' foi criado com {len(cells)} células.")
    else:
        print("Erro: Não foi possível extrair nenhuma célula do HTML.")

# Execução
converter_html_para_ipynb_v2('Lab01I_IAENG.html', 'Lab01I_IAENG.ipynb')

Sucesso! O arquivo 'Lab01I_IAENG.ipynb' foi criado com 7 células.
